# SHA-256 Basics
A hash function takes any input and turns it into a fixed-size "fingerprint". SHA-256 always gives 256 bits (32 bytes), no matter how big or small the input is.

## Step 1: Hash a message
`hashlib` works on bytes, not text, so we encode the string first. If you pass a plain string, Python gives a `TypeError`.

In [ ]:
import hashlib

message = "Hello, Khemshield"

# encode turns the text into bytes
message_bytes = message.encode("utf-8")
message_hash = hashlib.sha256(message_bytes)

# hashlib.new("sha256", message_bytes) does the same, choosing the algorithm by name

print("Message:", message)
print("Message Bytes:", message_bytes)

# This prints the hash object itself, not the result. Use .digest() or .hexdigest() to get the result.
print("Message Hash:", message_hash)

Message: Hello, Khemshield
Message Bytes: b'Hello, Khemshield'
Message Hash: <sha256 _hashlib.HASH object @ 0x00000179F30ACCF0>


## Step 2: digest and hexdigest
- `.digest()` gives the raw 32 bytes, which is what computers usually store or compare.
- `.hexdigest()` gives a readable string of 64 characters. Each hex character holds 4 bits, so 64 × 4 = 256 bits.

In [2]:
message_hash_digest_raw = message_hash.digest()
message_hash_digest_readable = message_hash.hexdigest()

print("Digest (raw):", message_hash_digest_raw)
print("Length:", len(message_hash_digest_raw), "bytes")

print("Digest (readable):", message_hash_digest_readable)
print("Length:", len(message_hash_digest_readable), "hex characters")

Digest (raw): b'\xd8\x93Q\xafw4+\xa0\xca\xb8;\x9e\xf0\x87N\xa8\xd3>\xec\xe1t\xee\x84.\xb6[\x02*\xb1\x95\xc8I'
Length: 32 bytes
Digest (readable): d89351af77342ba0cab83b9ef0874ea8d33eece174ee842eb65b022ab195c849
Length: 64 hex characters


## Step 3: The b"" literal
Putting `b` in front of the quotes makes bytes directly. This only works for plain English (ASCII) text. It gives the same hash as using `.encode("utf-8")`.

In [3]:
print(hashlib.sha256(b"Hello, Khemshield").hexdigest())
print(hashlib.sha256(b"Hello, Khemshield").hexdigest() == message_hash_digest_readable)

d89351af77342ba0cab83b9ef0874ea8d33eece174ee842eb65b022ab195c849
True


## Step 4: A helper function
A small helper saves typing `.encode()` every time. Running it twice on the same text always gives the same hash.

In [4]:
def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

print(sha256_text("Love Me Please"))
print(sha256_text("Love Me Please"))
print(sha256_text("Love Me Please") == sha256_text("Love Me Please"))

ac46aa07898233bf2c4820645ef0f456f1cc156ecefad7d5bf607a5fb0c3a95e
ac46aa07898233bf2c4820645ef0f456f1cc156ecefad7d5bf607a5fb0c3a95e
True


## Step 5: Verify a hash
To check that a message hasn't changed, hash it again and compare the result with the hash you expected. If they match, the message is exactly the same. If even one character changed, they won't match.

In [7]:
import hmac

# The hash we were given earlier (for example, published by the sender)
expected_hash = sha256_text("Hello, Khemshield")

def verify_text(text: str, expected: str) -> bool:
    actual = sha256_text(text)
    # compare_digest compares safely, which matters later for HMAC
    return hmac.compare_digest(actual, expected.strip().lower())

print(verify_text("Hello, Khemshield", expected_hash))   # True: unchanged
print(verify_text("Hello, Khemshield!", expected_hash))  # False: one character added
print(verify_text("hello, Khemshield", expected_hash))   # False: one letter changed case

True
False
False


You can check that `actual` is always clean:

In [8]:
actual = sha256_text("Hello, Khemshield")

print(actual == actual.lower())   # True: already lowercase
print(actual == actual.strip())   # True: no extra spaces
print(len(actual))                # 64

# What a copied hash might look like
copied = actual.upper() + "\n"
print(repr(copied))
print(hmac.compare_digest(actual, copied))                   # False: same hash, written differently
print(hmac.compare_digest(actual, copied.strip().lower()))   # True after cleaning

True
True
64
'D89351AF77342BA0CAB83B9EF0874EA8D33EECE174EE842EB65B022AB195C849\n'
False
True


### What verifying a hash does not prove
A matching hash proves the message didn't change. It does not prove who sent it. If an attacker can change the message, they can also just give you a new hash for it. To prove who sent something, we need a secret key (HMAC) or a private key (digital signatures).

## Summary
- A hash is a fixed-size fingerprint. SHA-256 always gives 32 bytes, whatever the input size.
- Text goes in as bytes, and results come out as bytes (`.digest()`) unless you ask for a readable version (`.hexdigest()`).
- The same input always gives the same hash.
- Never name a file `hashlib.py`. It hides the real library and breaks the import.
- To verify, hash again and compare. A match proves the data is unchanged, but not who sent it.